# NB-05 — LangChain Chains

**Learning goal:** Wrap the extraction logic into a LangChain chain, then build a second chain that generates a compliance-style confirmation message. Chain them together so extraction flows directly into confirmation.

**Concepts covered**
- `ChatAnthropic` LangChain wrapper
- `PromptTemplate` and `ChatPromptTemplate`
- `LLMChain` (and the modern `|` pipe syntax)
- Chaining two steps: extraction → confirmation
- Output parsers

## Cell 1 — Setup

In [ ]:
from dotenv import load_dotenv
import os
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal
import json

load_dotenv()

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    max_tokens=512
)

## Cell 2 — Extraction Prompt Template

In [ ]:
EXTRACTION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
Extract the five regulatory intake fields from the user's message.
Return ONLY a JSON object with these exact keys:
query_type, regulation_ref, product_area, urgency, submitting_team.

RULES:
- Never infer urgency from tone. Use "routine" if deadline is unclear.
- submitting_team must be a team name, never an individual's name.
- Use "other" for regulation_ref if the framework is not clear.

EXAMPLE:
Input: "PV team here. SUSAR for Phase III trial, EMA deadline 15 days per ICH E2A."
Output: {{"query_type": "safety_signal", "regulation_ref": "ICH_E2A",
  "product_area": "clinical", "urgency": "critical", "submitting_team": "PV Team"}}
"""

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM),
    ("human", "{user_message}")
])

## Cell 3 — Build Extraction Chain

In [ ]:
def parse_json_output(text: str) -> dict:
    """Strip markdown fences and parse JSON from LLM output."""
    clean = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(clean)

extraction_chain = extraction_prompt | llm | StrOutputParser()

raw_output = extraction_chain.invoke({
    "user_message": "CMC Regulatory here. FDA issued a 483 on our Pune site. 15 business days to respond."
})
print("Raw output:")
print(raw_output)

extracted = parse_json_output(raw_output)
print("\nParsed dict:")
print(extracted)

## Cell 4 — Confirmation Prompt Template

In [ ]:
CONFIRMATION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
Given a structured intake record, generate a brief formal confirmation
message for the submitter to review before the record is saved.

Format:
- Reference the query type and regulation formally
- State the urgency tier
- Name the submitting team
- End with: "Please confirm or correct before this record is saved."

Keep it under 60 words. Professional tone.
"""

confirmation_prompt = ChatPromptTemplate.from_messages([
    ("system", CONFIRMATION_SYSTEM),
    ("human", "Intake record: {intake_record}")
])

confirmation_chain = confirmation_prompt | llm | StrOutputParser()

## Cell 5 — Chain Extraction into Confirmation

In [ ]:
def run_extraction_and_confirmation(user_message: str):
    # Step 1: extract
    raw = extraction_chain.invoke({"user_message": user_message})
    extracted = parse_json_output(raw)
    print("Extracted fields:")
    print(json.dumps(extracted, indent=2))

    # Step 2: confirm
    confirmation = confirmation_chain.invoke({"intake_record": json.dumps(extracted)})
    print("\nConfirmation message:")
    print(confirmation)

    return extracted, confirmation

run_extraction_and_confirmation(
    "Labelling team. We need to update the SmPC for our oncology product "
    "following the latest EMA assessment report."
)

## Cell 6 — Experiment

In [ ]:
# Try these inputs and observe how the chains handle them:
inputs = [
    "PV team. SUSAR for NCT-2244, EMA report due in 15 days, ICH E2A.",
    "Hi we need to file a variation for our cardiovascular product with EMA.",
    "Can you check the weather in Mumbai?"
]

for inp in inputs:
    print(f"\nInput: {inp}")
    print("-" * 60)
    run_extraction_and_confirmation(inp)

### Key Takeaway

LangChain chains let you compose LLM steps cleanly. The `|` pipe syntax means each step's output becomes the next step's input. A two-step chain (extract → confirm) is the core of SmartIntake's LangChain layer.